# EXP-001 hybrid 검색 dev-small 확인

목적: dense + BM25(Kiwi) RRF 결합이 동작하고 방향이 맞는지 dev-small에서 본다. 판정은 dev-full로만 한다(`docs/experiments/EXP-001.md`). dev-small 점수는 dev-full과 비교하지 않는다.

In [1]:
from pathlib import Path

import pandas as pd

from rag.eval.metrics import gold_ranks, load_cfg, load_gold, min_docs, per_query, read_run

In [2]:
ROOT = Path.cwd().parent
LAYER = "dev-small"
RUNS = {
    "dense": ROOT / "data/runs/kure-v1-fixed-512-64-dev-small.csv",
    "hybrid": ROOT / "data/runs/kure-v1-fixed-512-64-hybrid-dev-small.csv",
}

## 순위 파일

순위 파일은 `uv run python -m rag.search +exp=exp001`로 만들었다. 질의마다 서로 다른 문서 10개가 있는지 본다.

In [3]:
runs = {name: read_run(path) for name, path in RUNS.items()}
for name, run in runs.items():
    per_q = run.groupby("qid")["doc_id"].nunique()
    print(name, run.shape, "질의", per_q.size, "질의당 문서 수", per_q.min(), "~", per_q.max())

dense (10390, 3) 질의 1039 질의당 문서 수 10 ~ 10
hybrid (10390, 3) 질의 1039 질의당 문서 수 10 ~ 10


## 지표 비교

In [4]:
cfg = load_cfg()
ev = cfg.eval
gold = load_gold(cfg, LAYER)
n_min = min_docs(ev.recall_ks, ev.mrr_k)
ranks = {name: gold_ranks(run, gold, n_min) for name, run in runs.items()}
scores = {name: per_query(r, ev.recall_ks, ev.mrr_k) for name, r in ranks.items()}
pd.DataFrame({name: s.mean() for name, s in scores.items()}).round(4)

,dense,hybrid
recall@1,0.8739,0.9105
recall@5,0.9673,0.9808
recall@10,0.9817,0.9875
mrr@10,0.9151,0.9438


In [5]:
pd.DataFrame({
    name: s.join(gold["meeting_name"]).groupby("meeting_name")["recall@5"].mean()
    for name, s in scores.items()
}).assign(n=gold["meeting_name"].value_counts()).round(4)

,dense,hybrid,n
meeting_name,,,
국정감사,0.9797,0.9913,345
본회의,0.9579,0.9789,95
소위원회,0.9612,0.9726,438
예산결산특별위원회,0.9286,0.9643,56
특별위원회,0.9810,0.9905,105


## 상위 5위 진입·이탈

RRF가 dense의 5위 안 정답을 밀어낸 질의(이탈)와 5위 밖 정답을 끌어올린 질의(진입)를 센다.

In [6]:
hit = pd.DataFrame({name: s["recall@5"] for name, s in scores.items()})
pd.crosstab(hit["dense"], hit["hybrid"], rownames=["dense@5"], colnames=["hybrid@5"])

hybrid@5,0.0,1.0
dense@5,,
0.0,17,17
1.0,3,1002


In [7]:
moved = pd.DataFrame(ranks).rename(columns=lambda c: f"rank_{c}")
moved[(moved.rank_dense <= 5) != (moved.rank_hybrid <= 5)].describe()

,rank_dense,rank_hybrid
count,14.000000,18.000000
mean,7.214286,2.555556
std,3.092884,1.423427
min,1.000000,1.000000
25%,6.250000,2.000000
50%,8.000000,2.000000
75%,9.750000,3.000000
max,10.000000,6.000000
